# 02 Portfolio Optimization

## A. 本步目标
在一个历史估计窗口上比较 equal-weight、unconstrained MV 和 carbon-constrained MV 的权重。

## B. 概念解释
Mean-variance 优化使用预期收益和 covariance matrix。Carbon constraint 是对合成分数加权平均值的上限，不代表真实排放或现实世界 impact。

## C. 本步默认值
使用最近 252 个收益观察、long-only、35% weight cap、risk aversion 5，以及 equal-weight carbon exposure 的 70% 作为预算。

In [ ]:
from pathlib import Path
import sys, yaml, pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from carbon_portfolio.data import download_adjusted_close, calculate_returns
from carbon_portfolio.carbon import carbon_scores_as_of, load_carbon_intensity, portfolio_carbon_exposure
from carbon_portfolio.optimization import equal_weights, optimize_mean_variance
cfg = yaml.safe_load((ROOT / 'config/settings.yaml').read_text())
prices = download_adjusted_close(cfg['data']['tickers'], cfg['data']['start_date'], cfg['data']['end_date'], ROOT / cfg['data']['cache_path'])
returns = calculate_returns(prices)
carbon_panel = load_carbon_intensity(ROOT / 'data/carbon_intensity.csv', returns.columns)
scores = carbon_scores_as_of(carbon_panel, returns.index[-1], returns.columns)
window = returns.tail(cfg['backtest']['lookback_days'])
mu, cov = window.mean() * 252, window.cov() * 252
equal = equal_weights(returns.columns)
budget = portfolio_carbon_exposure(equal, scores) * cfg['optimization']['carbon_budget_ratio']
unconstrained = optimize_mean_variance(mu, cov, cfg['optimization']['risk_aversion'], cfg['optimization']['max_weight'])
constrained = optimize_mean_variance(mu, cov, cfg['optimization']['risk_aversion'], cfg['optimization']['max_weight'], scores, budget)
pd.DataFrame({'equal_weight': equal, 'unconstrained_mv': unconstrained.weights, 'carbon_constrained_mv': constrained.weights})

## E. 自检
- 权重是否和为 1、非负且不超过上限？
- Carbon-constrained exposure 是否低于预算？
- 权重是否集中在边界？若是，应报告而不是隐藏。
- 历史均值是否被误称为可靠预测？不应这样表述。

## F. 向 Senior Quant 解释
本步展示相同估计输入下，线性 carbon budget 如何改变 mean-variance 权重。Carbon score 是教学 proxy，约束结果只能解释为模型内 exposure 的变化。单期最优权重可能对均值、协方差和 risk-aversion 高度敏感，尚不能说明策略表现或可交易性。